# Допуск: LongMemEval-V2: Evaluating Long-Term Agent Memory Toward Experienced Colleagues

**Статья:** D. Wu, Z. Ji, A. Kawatkar, B. Kwan, J.-C. Gu, N. Peng, K.-W. Chang, 2026 —
<https://arxiv.org/abs/2605.12493>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы, опираясь на код и статью.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / рисунок / приложение / цитата) нарушено и как должно быть. Код писать не
нужно — достаточно объяснить словами. Запускать и отлаживать код тоже не нужно: все задания
решаются чтением кода и статьи.

Общие импорты:

In [ ]:
import collections
import json
import shutil
import time
import torch

Траектория — один прогон веб-агента: цель (`goal` / `task`), исход (`success` / `failure`) и
последовательность состояний; у состояния есть URL, действие, дерево доступности (AXTree) и
скриншот (§3.2, прил. A.1). Стог (haystack) — упорядоченный список траекторий, который система
памяти получает для вопроса (§3.3). Функции, которые используются, но не показаны, считайте
корректными.

## Задание 1. Пять видов вопросов

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

Вопросы LME-V2 пишут люди (§3.2). Чтобы показать, какой опыт проверяет каждый вид вопроса,
функции ниже строят вопрос и ответ по траектории так, как это сделал бы аннотатор (примеры —
по рис. 1). Определите, какая функция соответствует каждой из пяти способностей: static state
recall, dynamic state tracking, workflow knowledge, environment gotchas, premise awareness
(§3.1). Ошибок в коде нет.

In [ ]:
def make_a(traj, i, j):
    # states i and j: two different forms seen in the trajectory history
    a, b = traj.states[i], traj.states[j]
    extra = sorted(set(a.buttons) - set(b.buttons))
    return f"Which button does the {a.page} form have that the {b.page} form does not?", extra

def make_b(traj, i):
    before, after = traj.states[i], traj.states[i + 1]
    new = sorted(set(after.fields) - set(before.fields))
    return f"On the {before.page} page, after I {traj.actions[i]}, which field appears?", new

def make_c(traj):
    modules = [s.module for s in traj.states if s.module]
    return f"Which modules, in order, does our workflow use to {traj.task}?", list(dict.fromkeys(modules))

def make_d(traj, i, cause):
    # traj.outcome == "failure"; cause: what an annotator understood after reading the whole run
    msg = (f"Hi, I'm new here. I tried to {traj.task}, but got the screen in the attached image. "
           "What is the most likely explanation?")
    return {"text": msg, "image": traj.states[i].screenshot}, cause

def make_e(question, true_detail, wrong_detail):
    # question: an answerable question built by one of the other functions
    flawed = question.replace(true_detail, wrong_detail)
    return flawed, f"No such thing here: the question assumes {wrong_detail}, but it is {true_detail}"

**Вопросы.**
1. Из вопросов каких видов строятся вопросы `make_e` и сколько их в LME-V2 (рис. 2)? В табл. 2
   нет отдельного столбца для этой способности. Куда попадают такие вопросы? Проверьте на
   строке AgentRunbook-C (LME-V2-Small), как столбец Overall получается из четырёх столбцов
   по категориям, — и почему это не простое среднее четырёх чисел.
2. Почему вопросы `make_d` оформлены как сообщение новичка со скриншотом и строятся по
   неудачной траектории (§3.2, рис. 1)? Зачем в стогах держат примерно поровну успешных и
   неудачных траекторий? Что показывают табл. 2 и рис. 4 о трудности этой категории даже
   при идеальном поиске?
3. Траектории, где есть ответ на вопрос, в среднем занимают 1.39 траектории и 310.8K токенов
   (рис. 3). Что показывает второе пилотное исследование (§3.4, рис. 4, прил. B.2): хватает ли
   читателю самих этих траекторий, что дают срезы с заметками и Codex? Как это объясняет,
   почему статья оценивает память через компактный возвращаемый контекст и почему в итоге
   строит память на основе кодового агента?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Сборка стогов

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Сборка стогов одного домена (прил. A.4) по готовой карте покрытия (`coverage`) и уже
выбранному минимальному ядру ответов (`seed_of`). Упрощения: у вопроса одна «ступень»
доказательства, ранжирование по разнообразию задач опущено, вопросы на отказ без
подтверждающих траекторий не рассматриваются. `rng` — `random.Random`. Найдите одно
отклонение от статьи.

In [ ]:
def build_haystacks(questions, coverage, seed_of, pool, outcome, rng, small_size=100, medium_size=500):
    # coverage[q]: ids of ALL trajectories in the pool that contain the answer to q
    # seed_of[q]: answer trajectories picked for q by the minimal-answer-core assignment
    core = sorted(set().union(*(seed_of[q] for q in questions)))
    support = collections.Counter(t for q in questions for t in coverage[q])   # questions answered by t

    def fillers(cands, k):                          # outcome-balanced, low answer-support first
        succ = [t for t in cands if outcome[t] == "success"]
        fail = [t for t in cands if outcome[t] == "failure"]
        rng.shuffle(succ); rng.shuffle(fail)
        succ.sort(key=support.__getitem__); fail.sort(key=support.__getitem__)
        return succ[:k // 2] + fail[:k - k // 2]

    small = core + fillers([t for t in pool if t not in core], small_size - len(core))
    rng.shuffle(small)                              # one haystack shared by all questions of the domain

    medium = {}
    for q in questions:
        seed = list(seed_of[q])                     # same answer seed as in the small tier
        cands = [t for t in pool if t not in seed]  # any other trajectory may serve as a filler
        hay = seed + fillers(cands, medium_size - len(seed))
        rng.shuffle(hay)
        medium[q] = hay
    return small, medium

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Прогон одного вопроса

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Оценка системы памяти на одном вопросе в постановке context gathering (§3.3, прил. A.5).
`make_memory()` создаёт систему памяти с методами `insert` / `query`, `reader` — вызов
фиксированного читателя, `judge` — LLM-судья, `deterministic_match` — проверка по
`eval_function` вопроса, `truncate` обрезает контекст по токенам, `last_boxed` достаёт
последнее выражение `\boxed{}`. Найдите одно отклонение от статьи.

In [ ]:
ENV = {"enterprise": "working in a customized ServiceNow environment",
       "web": "in a web browsing environment with customized Magento shop, Magento admin and Postmill forum sites"}
READER_SYSTEM = {d: f"You are an experienced colleague {env}. Answer based on your memory of the environment. "
                    "If you are not sure, give your best guess. If you believe the question's "
                    "construction/premise is wrong, explain in \\boxed{} why the question is flawed."
                 for d, env in ENV.items()}

def evaluate_question(make_memory, q, haystack, reader, judge, tok, budget=200_000):
    memory = make_memory()
    for traj in haystack:                        # stream the history in its fixed order
        memory.insert(traj)
    t0 = time.time()
    context = memory.query(q["question"], q["image"])      # list of text / image items
    latency = time.time() - t0
    context = truncate(context, tok, budget)               # Qwen3.5-9B tokens
    user = ["### Memory context:\n", *context, "\n\n### Question to answer:\n" + q["question"], q["image"]]
    response = reader(READER_SYSTEM[q["domain"]], user)    # fixed reader: Qwen3.5-9B
    pred = last_boxed(response)                  # the whole response if there is no \boxed{}
    if pred.strip().lower() == "unknown":
        correct = False
    elif q["eval"] in ("abstention", "gotchas"):
        correct = judge(q, response, pred)       # GPT-5.2, binary label
    else:
        correct = deterministic_match(q, pred)   # normalized phrases / choice letters
    return correct, latency

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Запрос к AgentRunbook-R

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Этап запроса AgentRunbook-R (§4.1, прил. C.2). Пулы памяти уже построены при вставке
траекторий; `controller` — LLM-контроллер (Qwen3.5-9B) с промптом из табл. 9, возвращает JSON
с запросами к пулам; `embed` — Qwen3-Embedding-8B, вектор нормирован.

In [ ]:
def runbook_r_query(question, controller, embed, pools):
    # pools[name] = (entries, E): entries built at insertion time, E: (N, D) L2-normalised embeddings
    # "raw": radius-1 state slices, "event": state transitions, "note": procedure / hint notes
    plan = json.loads(controller(question, pools))     # a stream may be left empty to skip it
    raw_qs = list(dict.fromkeys(plan.get("raw_state_queries") or []))[:5]
    m = min(2, 6 // len(raw_qs)) if raw_qs else 0      # results kept per raw-state query

    def search(name, query, k):
        entries, E = pools[name]
        sims = E @ embed(query)                        # (N,)
        return [entries[i] for i in sims.topk(min(k, len(entries))).indices.tolist()]

    slices = []
    for q in raw_qs:
        for e in search("raw", q, m):
            if all(e["state_id"] != s["state_id"] for s in slices):   # merge duplicate states
                slices.append(e)
    events = search("event", plan["event_query"], 6) if plan.get("event_query") else []
    notes = search("note", plan["note_query"], 3) if plan.get("note_query") else []
    return notes + events + slices                     # rendered in this order, with screenshots

**Вопросы.**
1. Для каких видов вопросов нужен каждый из трёх пулов? Опираясь на абляции в табл. 2,
   скажите, без какого пула сильнее всего проседают вопросы static и почему; что происходит с
   workflow-вопросами, если оставить только локальные наблюдения (§5.1)?
2. Чем этот запрос отличается от базового RAG «query → slice + notes» (прил. C.1)? За счёт
   чего AgentRunbook-R отвечает ~26 с против 0.2 с и что даёт эта цена (табл. 2, рис. 6, §5.2)?
   Почему построение событий — отдельный вызов LLM на каждый переход между состояниями — не
   входит в эту задержку?
3. AgentRunbook-R делает меньше ошибок поиска и чтения, чем RAG с заметками, но вопросы на
   ложную посылку решает не лучше (рис. 7, прил. D.1). Почему, если смотреть на то, что
   возвращает эта функция? Что для этого делает AgentRunbook-C?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Запрос к AgentRunbook-C

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Этап запроса AgentRunbook-C (§4.2, прил. C.3, табл. 10–11). `run_agent` запускает кодового
агента в песочнице; `render_manifests`, `is_valid_span`, `render_states` — вспомогательные
функции обвязки. Найдите один неверный комментарий.

In [ ]:
def runbook_c_query(question, store, sandbox, run_agent, assets, span_budget=20):
    # store: directory written by Insert, one trajectory.json + screenshots/ per trajectory
    sandbox.mkdir(parents=True)
    (sandbox / "question.json").write_text(json.dumps(question))
    shutil.copy(assets / "INSTRUCTION.md", sandbox)                 # workflow document
    shutil.copytree(assets / "scripts", sandbox / "scripts")        # inspect_trajectory.py helper
    (sandbox / "trajectories").symlink_to(store)
    render_manifests(store)            # TRAJECTORY_SUMMARY_CONCISE.md / _FULL.md for the current haystack

    t0 = time.time()
    run_agent(sandbox)                 # `codex exec`, GPT-5.4-mini; writes memory_module_output.json
    latency = time.time() - t0
    # The scaffolding buys accuracy at the price of speed: on top of its usual raw exploration
    # the agent now also reads the manifests and calls the helper, so it runs more commands
    # per question and answers more slowly than off-the-shelf Codex.

    out = json.loads((sandbox / "memory_module_output.json").read_text())
    spans, used = [], 0
    for s in out["trajectory_spans"]:                               # ordered by importance
        n = s["end_state_index"] - s["start_state_index"] + 1       # zero-based, inclusive
        if is_valid_span(s, store) and used + n <= span_budget:
            spans.append(s)
            used += n
    return [out["memory_markdown"]] + render_states(spans, store), latency

**Ваш ответ:**

_Напишите ответ здесь._